# 02 - Data Cleaning: Preparing Tour de France Data

This notebook focuses on cleaning and standardizing the raw data collected in the web scraping phase.

## Objectives
- Load raw scraped data
- Standardize names, dates, and times
- Handle missing data
- Convert relative times to absolute timestamps
- Save cleaned data for analysis

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import json

## Setup Data Paths

In [2]:
# Define paths
raw_data_dir = Path('data/raw')
cleaned_data_dir = Path('data/cleaned')
cleaned_data_dir.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {raw_data_dir.absolute()}")
print(f"Cleaned data directory: {cleaned_data_dir.absolute()}")

Raw data directory: c:\DataSciencePPTS\bike_race_analysis\data\raw
Cleaned data directory: c:\DataSciencePPTS\bike_race_analysis\data\cleaned


## Part 1: Load Raw Data

We'll load the data we scraped in the previous notebook.

In [3]:
# Load Wikipedia races data
wiki_races_df = pd.read_csv(raw_data_dir / 'wikipedia_races.csv')
print(f"Loaded {len(wiki_races_df)} Wikipedia race records")
print("\nFirst few rows:")
wiki_races_df.head()

Loaded 10 Wikipedia race records

First few rows:


,name,country,type
0,Tour de France,France,Grand Tour
1,Giro d'Italia,Italy,Grand Tour
2,Vuelta a España,Spain,Grand Tour
3,Paris–Roubaix,France,Monument
4,Milan–San Remo,Italy,Monument


In [4]:
# Load Tour de France stages data
tour_stages_df = pd.read_csv(raw_data_dir / 'tour_stages.csv')
print(f"Loaded {len(tour_stages_df)} Tour de France stage records")
print("\nFirst few rows:")
tour_stages_df.head()

Loaded 10 Tour de France stage records

First few rows:


,year,stage,date,start,finish,distance_km,type,winner,time
0,2023,1,2023-07-01,Bilbao,Bilbao,182.0,Flat,Adam Yates,04:18:42
1,2023,2,2023-07-02,Vitoria-Gasteiz,San Sebastián,209.0,Hilly,Victor Lafay,04:51:23
2,2023,3,2023-07-03,Amorebieta-Etxano,Bayonne,193.5,Flat,Biniam Girmay,04:32:18
3,2023,4,2023-07-04,Dax,Nogaro,182.5,Flat,Jasper Philipsen,04:03:31
4,2023,5,2023-07-05,Pau,Laruns,163.0,Mountain,Jai Hindley,04:22:19


## Part 2: Data Exploration and Quality Check

Let's examine the data quality and identify issues.

In [5]:
# Check for missing values in Wikipedia races
print("Missing values in Wikipedia races data:")
print(wiki_races_df.isnull().sum())
print("\nData types:")
print(wiki_races_df.dtypes)

Missing values in Wikipedia races data:
name       0
country    0
type       0
dtype: int64

Data types:
name       object
country    object
type       object
dtype: object


In [6]:
# Check for missing values in Tour stages
print("Missing values in Tour stages data:")
print(tour_stages_df.isnull().sum())
print("\nData types:")
print(tour_stages_df.dtypes)
print("\nBasic statistics:")
print(tour_stages_df.describe())

Missing values in Tour stages data:
year           0
stage          0
date           0
start          0
finish         0
distance_km    0
type           0
winner         0
time           0
dtype: int64

Data types:
year             int64
stage            int64
date            object
start           object
finish          object
distance_km    float64
type            object
winner          object
time            object
dtype: object

Basic statistics:
              year      stage  distance_km
count    10.000000  10.000000    10.000000
mean   2022.500000   3.000000   165.620000
std       0.527046   1.490712    55.964667
min    2022.000000   1.000000    13.200000
25%    2022.000000   2.000000   165.125000
50%    2022.500000   3.000000   182.000000
75%    2023.000000   4.000000   190.750000
max    2023.000000   5.000000   209.000000


## Part 3: Clean Wikipedia Races Data

We'll standardize the Wikipedia races data.

In [7]:
def clean_wikipedia_races(df):
    """
    Clean and standardize Wikipedia races data.
    """
    df_clean = df.copy()
    
    # Standardize column names (lowercase with underscores)
    df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')
    
    # Remove leading/trailing whitespace from string columns
    for col in df_clean.select_dtypes(include=['object']).columns:
        df_clean[col] = df_clean[col].str.strip()
    
    # Replace 'N/A' with actual NaN
    df_clean.replace('N/A', np.nan, inplace=True)
    
    # Drop rows with missing race names (critical field)
    df_clean = df_clean.dropna(subset=['name'])
    
    # Fill missing country/type with 'Unknown'
    df_clean['country'] = df_clean['country'].fillna('Unknown')
    df_clean['type'] = df_clean['type'].fillna('Unknown')
    
    # Remove duplicates
    df_clean = df_clean.drop_duplicates(subset=['name'], keep='first')
    
    print(f"Cleaned Wikipedia races: {len(df)} → {len(df_clean)} records")
    return df_clean

wiki_races_clean = clean_wikipedia_races(wiki_races_df)
wiki_races_clean.head()

Cleaned Wikipedia races: 10 → 10 records


,name,country,type
0,Tour de France,France,Grand Tour
1,Giro d'Italia,Italy,Grand Tour
2,Vuelta a España,Spain,Grand Tour
3,Paris–Roubaix,France,Monument
4,Milan–San Remo,Italy,Monument


## Part 4: Clean Tour de France Stages Data

This is where we'll do more intensive cleaning:
- Standardize dates
- Convert time strings to proper format
- Handle missing data
- Add calculated fields

In [8]:
def parse_time_to_seconds(time_str):
    """
    Convert time string (HH:MM:SS) to total seconds.
    """
    if pd.isna(time_str):
        return np.nan
    
    parts = str(time_str).split(':')
    if len(parts) == 3:
        hours, minutes, seconds = map(int, parts)
        return hours * 3600 + minutes * 60 + seconds
    return np.nan

def seconds_to_time_str(seconds):
    """
    Convert seconds to HH:MM:SS format.
    """
    if pd.isna(seconds):
        return np.nan
    
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

In [9]:
def clean_tour_stages(df):
    """
    Clean and standardize Tour de France stages data.
    """
    df_clean = df.copy()
    
    # Standardize column names
    df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')
    
    # Convert date to datetime
    df_clean['date'] = pd.to_datetime(df_clean['date'])
    
    # Remove leading/trailing whitespace from string columns
    for col in df_clean.select_dtypes(include=['object']).columns:
        df_clean[col] = df_clean[col].str.strip()
    
    # Ensure distance is numeric
    df_clean['distance_km'] = pd.to_numeric(df_clean['distance_km'], errors='coerce')
    
    # Convert time to seconds for easier calculations
    df_clean['time_seconds'] = df_clean['time'].apply(parse_time_to_seconds)
    
    # Calculate average speed (km/h)
    df_clean['avg_speed_kmh'] = np.where(
        df_clean['time_seconds'] > 0,
        (df_clean['distance_km'] / df_clean['time_seconds']) * 3600,
        np.nan
    )
    
    # Round to 2 decimal places
    df_clean['avg_speed_kmh'] = df_clean['avg_speed_kmh'].round(2)
    
    # Standardize stage type categories
    stage_type_mapping = {
        'flat': 'Flat',
        'hilly': 'Hilly',
        'mountain': 'Mountain',
        'time trial': 'Time Trial',
        'tt': 'Time Trial'
    }
    df_clean['type'] = df_clean['type'].str.lower().map(stage_type_mapping).fillna(df_clean['type'])
    
    # Drop rows with missing critical data
    df_clean = df_clean.dropna(subset=['year', 'stage', 'date'])
    
    # Sort by year and stage
    df_clean = df_clean.sort_values(['year', 'stage']).reset_index(drop=True)
    
    print(f"Cleaned Tour stages: {len(df)} → {len(df_clean)} records")
    return df_clean

tour_stages_clean = clean_tour_stages(tour_stages_df)
tour_stages_clean.head()

Cleaned Tour stages: 10 → 10 records


,year,stage,date,start,finish,distance_km,type,winner,time,time_seconds,avg_speed_kmh
0,2022,1,2022-07-01,Copenhagen,Copenhagen,13.2,Time Trial,Yves Lampaert,00:15:17,917,51.82
1,2022,2,2022-07-02,Roskilde,Nyborg,202.5,Flat,Fabio Jakobsen,04:35:28,16528,44.11
2,2022,3,2022-07-03,Vejle,Sønderborg,182.0,Flat,Dylan Groenewegen,04:16:34,15394,42.56
3,2022,4,2022-07-05,Dunkerque,Calais,171.5,Flat,Wout van Aert,03:52:30,13950,44.26
4,2022,5,2022-07-06,Lille,Arenberg,157.0,Hilly,Simon Clarke,03:13:35,11615,48.66


## Part 5: Data Quality Verification

Let's verify our cleaned data.

In [10]:
# Check cleaned Tour stages data
print("Cleaned Tour stages data info:")
print(tour_stages_clean.info())
print("\nMissing values:")
print(tour_stages_clean.isnull().sum())
print("\nSummary statistics:")
print(tour_stages_clean[['distance_km', 'time_seconds', 'avg_speed_kmh']].describe())

Cleaned Tour stages data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   year           10 non-null     int64         
 1   stage          10 non-null     int64         
 2   date           10 non-null     datetime64[ns]
 3   start          10 non-null     object        
 4   finish         10 non-null     object        
 5   distance_km    10 non-null     float64       
 6   type           10 non-null     object        
 7   winner         10 non-null     object        
 8   time           10 non-null     object        
 9   time_seconds   10 non-null     int64         
 10  avg_speed_kmh  10 non-null     float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(5)
memory usage: 1008.0+ bytes
None

Missing values:
year             0
stage            0
date             0
start            0
finish           0


In [11]:
# Display sample of cleaned data with new calculated fields
print("Sample cleaned data:")
tour_stages_clean[['year', 'stage', 'date', 'distance_km', 'type', 'winner', 'avg_speed_kmh']].head(10)

Sample cleaned data:


,year,stage,date,distance_km,type,winner,avg_speed_kmh
0,2022,1,2022-07-01,13.2,Time Trial,Yves Lampaert,51.82
1,2022,2,2022-07-02,202.5,Flat,Fabio Jakobsen,44.11
2,2022,3,2022-07-03,182.0,Flat,Dylan Groenewegen,42.56
3,2022,4,2022-07-05,171.5,Flat,Wout van Aert,44.26
4,2022,5,2022-07-06,157.0,Hilly,Simon Clarke,48.66
5,2023,1,2023-07-01,182.0,Flat,Adam Yates,42.21
6,2023,2,2023-07-02,209.0,Hilly,Victor Lafay,43.04
7,2023,3,2023-07-03,193.5,Flat,Biniam Girmay,42.64
8,2023,4,2023-07-04,182.5,Flat,Jasper Philipsen,44.97
9,2023,5,2023-07-05,163.0,Mountain,Jai Hindley,37.28


## Part 6: Save Cleaned Data

Finally, we'll save our cleaned data for analysis.

In [12]:
# Save cleaned Wikipedia races
wiki_races_clean.to_csv(cleaned_data_dir / 'wikipedia_races_clean.csv', index=False)
wiki_races_clean.to_json(cleaned_data_dir / 'wikipedia_races_clean.json', orient='records', indent=2)

print(f"Saved cleaned Wikipedia races data: {len(wiki_races_clean)} records")

Saved cleaned Wikipedia races data: 10 records


In [13]:
# Save cleaned Tour stages
# Convert datetime to string for JSON serialization
tour_stages_for_json = tour_stages_clean.copy()
tour_stages_for_json['date'] = tour_stages_for_json['date'].dt.strftime('%Y-%m-%d')

tour_stages_clean.to_csv(cleaned_data_dir / 'tour_stages_clean.csv', index=False)
tour_stages_for_json.to_json(cleaned_data_dir / 'tour_stages_clean.json', orient='records', indent=2)

print(f"Saved cleaned Tour stages data: {len(tour_stages_clean)} records")

Saved cleaned Tour stages data: 10 records


## Summary

In this notebook, we:
1. ✓ Loaded raw scraped data from CSV files
2. ✓ Explored data quality and identified issues
3. ✓ Standardized column names and data types
4. ✓ Handled missing data appropriately
5. ✓ Converted time strings to seconds for calculations
6. ✓ Added calculated fields (average speed)
7. ✓ Saved cleaned data in CSV and JSON formats

### Data Cleaning Highlights
- Converted dates to proper datetime format
- Parsed time strings and converted to seconds
- Calculated average speeds from distance and time
- Standardized categorical values (stage types)
- Removed duplicates and handled missing values

### Output Files
- `data/cleaned/wikipedia_races_clean.csv` - Cleaned race information
- `data/cleaned/wikipedia_races_clean.json` - Same data in JSON
- `data/cleaned/tour_stages_clean.csv` - Cleaned Tour de France stages
- `data/cleaned/tour_stages_clean.json` - Same data in JSON

### Next Steps
Proceed to `03_data_analysis.ipynb` to analyze the cleaned data and answer interesting questions about Tour de France performance.